# 2. Export to ONNX, upload to S3, register

ONNX makes the model portable: the same file runs in OpenVINO Model Server on OpenShift AI and on a MicroShift edge node.
Upload uses the **My Storage** connection attached to this workbench (its `AWS_*` environment variables).

In [ ]:
# Install the demo package from the cloned repo (skipped if already installed)
import importlib.util
if importlib.util.find_spec("bearing") is None:
    %pip install -q -e "..[notebook]"
    print("Installed. If imports fail, restart the kernel once.")

In [ ]:
import os, json, pickle
from bearing.model import export_onnx, verify_onnx
from bearing.simulate import make_dataset

with open("../models/bearing-sklearn.pkl", "rb") as f:
    clf = pickle.load(f)
metrics = json.load(open("../models/bearing-metrics.json"))
VERSION = "1"
path = export_onnx(clf, f"../models/bearing/{VERSION}/model.onnx")
X_check, _ = make_dataset(50, seed=99)
print("exported", path, "| max difference vs scikit-learn:", verify_onnx(path, clf, X_check))

In [ ]:
import onnx
m = onnx.load(path)
print("inputs: ", [(i.name, [d.dim_param or d.dim_value for d in i.type.tensor_type.shape.dim]) for i in m.graph.input])
print("outputs:", [(o.name, [d.dim_param or d.dim_value for d in o.type.tensor_type.shape.dim]) for o in m.graph.output])
print("operators:", sorted({n.op_type for n in m.graph.node}))

## Upload to S3 (layout OVMS expects: `models/bearing/<version>/model.onnx`)

In [ ]:
if os.environ.get("AWS_ACCESS_KEY_ID") and os.environ.get("AWS_S3_BUCKET"):
    from bearing.storage import upload_model
    MODEL_URI = upload_model(path, VERSION)
    print("uploaded:", MODEL_URI)
else:
    MODEL_URI = None
    print("No S3 connection attached to this workbench; skipping upload (attach 'My Storage' to enable).")

## Register version 1 in the Model Registry (optional)
Set `MODEL_REGISTRY_URL` to your registry's REST route. Skipped if not set.

In [ ]:
if MODEL_URI:
    from bearing.registry import register
    register(MODEL_URI, VERSION, metrics)

**Next:** deploy it. Dashboard: *Models > Deploy model* with the OpenVINO Model Server runtime, framework `onnx - 1`, connection *My Storage*, path `models/bearing/`. Or `oc apply -f openshift-ai/10-serving.yaml`.